# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subh-hub912/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row is one **content item snapshot** (`content_id`) with trailing performance aggregates. In this file the grain is content-level (30,000 unique content IDs) across 32 pseudonymized clients.

Time coverage is by relative windows, not calendar dates:
- `*_90d` columns summarize trailing 90-day activity.
- `*_last_30d` and `*_prev_30d` are nested sub-windows inside that broader history.

There is no `snapshot_date` column, so we can verify window logic but **cannot** pin metrics to explicit start/end calendar dates in this dataset.


In [ ]:
from pathlib import Path
import pandas as pd

CANDIDATE_PATHS = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv'),
]

for path in CANDIDATE_PATHS:
    if path.exists():
        DATA_PATH = path
        break
else:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv from this notebook location.')

df = pd.read_csv(DATA_PATH)

unit_check = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'unique_content_id': df['content_id'].nunique(),
    'duplicate_content_id_rows': int(df.duplicated(subset=['content_id']).sum()),
    'unique_client_id': df['client_id'].nunique(),
})

window_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
]

print('Loaded:', DATA_PATH)
display(unit_check.to_frame('value'))
display(df[window_cols].describe().T[['min', 'max', 'mean']].round(2))
print('Has explicit snapshot date column?', 'snapshot_date' in df.columns)


## 2. Fields: feature / label / context / excluded

I am treating this as a **refresh-prioritization classification** problem where the outcome is trend direction.

- **Label / proxy (never features):** `trend_direction`, `trend_pct`
- **Context (join/split/read, not model input):** `content_id`, `client_id`
- **Excluded:**
  - `trend_direction`, `trend_pct` from model features (direct leakage into the target).
  - Any future-only metric would be excluded, but none is explicitly timestamped here; this is a key dataset limit.
- **Feature candidates (known at snapshot time):**
  - Market/query context: `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`
  - Content descriptors: `content_type`, `word_count`, `char_count`, `provider_used`, `model_used`, `content_age_days`, `age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`
  - Performance history: `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`


In [ ]:
label_or_proxy = {'trend_direction', 'trend_pct'}
context = {'content_id', 'client_id'}

feature_candidates = {
    'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'provider_used', 'model_used',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier'
}

bucket_union = feature_candidates | label_or_proxy | context
unassigned_columns = sorted(set(df.columns) - bucket_union)
overlap_count = len(feature_candidates & label_or_proxy) + len(feature_candidates & context) + len(label_or_proxy & context)

summary = pd.Series({
    'features': len(feature_candidates),
    'label_or_proxy': len(label_or_proxy),
    'context': len(context),
    'total_bucketed': len(bucket_union),
    'dataset_columns': len(df.columns),
    'overlap_count_between_buckets': overlap_count,
    'unassigned_columns_count': len(unassigned_columns),
})

display(summary.to_frame('value'))
print('Unassigned columns:', unassigned_columns if unassigned_columns else 'None')
print('Leakage guard active? ', label_or_proxy.isdisjoint(feature_candidates))


## 3. Verify it with queries (grain, counts, missing values, windows)

Checks executed below:
1. **Grain check**: one row per `content_id` (no duplicate unit rows).
2. **Count checks**: total rows, unique content, and client coverage.
3. **Missingness checks**: overall missingness and pattern by `content_type` for high-impact fields.
4. **Window checks**: `days_with_*` bounded by 90 and 30+30 windows not exceeding 90-day totals.


In [ ]:
# 1) Grain + counts
grain_probe = df.groupby('content_id').size().reset_index(name='rows_per_content')
duplicates = grain_probe[grain_probe['rows_per_content'] > 1]

count_summary = pd.Series({
    'rows': len(df),
    'unique_content_id': df['content_id'].nunique(),
    'duplicate_content_id_rows': int(duplicates.shape[0]),
    'unique_client_id': df['client_id'].nunique(),
})
display(count_summary.to_frame('value'))

# 2) Missingness overall (top 10)
missing_overall = (df.isna().mean().sort_values(ascending=False).head(10) * 100).round(2)
display(missing_overall.to_frame('missing_pct'))

# 3) Missingness pattern by content_type for selected fields
check_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'provider_used', 'model_used']
missing_by_type = (
    df.groupby('content_type')[check_cols]
      .apply(lambda g: g.isna().mean() * 100)
      .round(2)
)
display(missing_by_type)

# 4) Window logic checks
window_checks = pd.Series({
    'days_with_impressions_outside_0_90': int(((df['days_with_impressions'] < 0) | (df['days_with_impressions'] > 90)).sum()),
    'days_with_sessions_outside_0_90': int(((df['days_with_sessions'] < 0) | (df['days_with_sessions'] > 90)).sum()),
    'impressions_last30_plus_prev30_exceeds_90d': int((df['impressions_last_30d'] + df['impressions_prev_30d'] > df['impressions_90d']).sum()),
    'clicks_last30_plus_prev30_exceeds_90d': int((df['clicks_last_30d'] + df['clicks_prev_30d'] > df['clicks_90d']).sum()),
    'sessions_last30_plus_prev30_exceeds_90d': int((df['sessions_last_30d'] + df['sessions_prev_30d'] > df['sessions_90d']).sum()),
})
display(window_checks.to_frame('count'))


## 4. Data limits

Observed limits from this dataset:
- **No explicit calendar date per row**: we can use relative windows, but cannot align outcomes to external events by date.
- **Snapshot-only structure**: this is not a raw daily panel, so within-window dynamics are compressed into aggregates.
- **Label leakage risk is structural**: `trend_direction` and `trend_pct` encode the target direction directly.
- **Missingness is patterned, not random**: several fields are absent by content type, so naive imputation can inject content-type signals.
- **Decision-support only**: this can rank refresh opportunities, but cannot prove causal impact of making a content change.


In [ ]:
limits_evidence = pd.Series({
    'has_snapshot_date_column': 'snapshot_date' in df.columns,
    'rows_with_null_trend_pct': int(df['trend_pct'].isna().sum()),
    'pct_rows_with_null_trend_pct': round(df['trend_pct'].isna().mean() * 100, 2),
    'distinct_trend_direction_values': df['trend_direction'].nunique(),
})
display(limits_evidence.to_frame('value'))

trend_null_pattern = (
    df.groupby('trend_direction')['trend_pct']
      .apply(lambda s: round(s.isna().mean() * 100, 2))
      .rename('trend_pct_missing_pct')
)
display(trend_null_pattern.to_frame())

print('Columns that contain explicit date information:', [c for c in df.columns if 'date' in c.lower()])


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
